In [ ]:
# Example 1: Inspect top-level public API surface
import agent_framework

public_names = [n for n in dir(agent_framework) if not n.startswith('_')]
print(f"agent_framework has {len(public_names)} public names")
print("Sample of public API:")
for name in public_names[:25]:
    obj = getattr(agent_framework, name)
    print(f"  {name}: {type(obj).__name__}")

assert len(public_names) > 0, "agent_framework should expose public API"


In [ ]:
# Example 2: Construct Messages and verify role/content
# agent_framework >= 1.10 renamed ChatMessage -> Message; roles are plain string
# literals ("system" | "user" | "assistant" | "tool"), not a Role enum.
from agent_framework import Message

msg_user = Message(role="user", contents=["Hello, agent!"])
msg_assistant = Message(role="assistant", contents=["Hi there!"])

print(f"User message role:      {msg_user.role}")
print(f"User message text:      {msg_user.text!r}")
print(f"Assistant message role: {msg_assistant.role}")
print(f"Assistant message text: {msg_assistant.text!r}")

assert msg_user.role == "user"
assert msg_assistant.role == "assistant"
assert msg_user.text == "Hello, agent!"
assert msg_assistant.text == "Hi there!"

In [ ]:
# Example 3: Define a FunctionTool via the @tool decorator
# (agent_framework >= 1.10 renamed ai_function -> tool; invoke() is async).
from agent_framework import FunctionTool, tool

@tool(description="Add two integers and return the sum.")
def add(a: int, b: int) -> int:
    return a + b

print(f"add tool name: {add.name}")
print(f"add tool description: {add.description}")
result = await add.invoke(arguments={"a": 2, "b": 3})
print(f"add(2, 3) -> {result!r}")

assert isinstance(add, FunctionTool)
assert add.name == "add"
assert add.description == "Add two integers and return the sum."
assert add.func(2, 3) == 5

In [ ]:
# Example 4: ChatOptions construction + merging
# (agent_framework >= 1.10 made ChatOptions a TypedDict — a plain dict at runtime;
# merge_chat_options is the documented way to layer overrides).
from agent_framework import ChatOptions, merge_chat_options

opts = ChatOptions(temperature=0.7, max_tokens=128, top_p=0.9)
print(f"temperature: {opts['temperature']}")
print(f"max_tokens:  {opts['max_tokens']}")
print(f"top_p:       {opts['top_p']}")

merged = merge_chat_options(opts, {"max_tokens": 256})
print(f"merged: {merged}")

assert isinstance(opts, dict)
assert opts["temperature"] == 0.7
assert merged["max_tokens"] == 256
assert merged["temperature"] == 0.7

In [ ]:
# Example 5: AgentSession for conversation state
# (agent_framework >= 1.10 replaced AgentThread with AgentSession).
from agent_framework import AgentSession, Message

session = AgentSession(session_id="smoke-session-1")
print(f"session type: {type(session).__name__}")
print(f"session_id: {session.session_id}")

restored = AgentSession.from_dict(session.to_dict())
print(f"round-tripped session_id: {restored.session_id}")

msg = Message(role="user", contents=["What is 2+2?"])
print(f"created message: role={msg.role} text={msg.text!r}")

assert session.session_id == "smoke-session-1"
assert restored.session_id == session.session_id
assert isinstance(msg, Message)
print("agent_framework_core smoke test PASSED")